<a href="https://colab.research.google.com/github/armro/CheckStrangsConjecture/blob/main/Check_StrangsConjecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description of Routines for Checking Strang's Conjecture

The code below will allow us to check if Strang's conjecture is true for a provided mesh. The user needs to give a mesh and polynomial order.



# Instructions:



*   Run the "Uploading NGSolve" cell
*   Run the "Main Code" cell
*   create a mesh
*   Run the code


There are several examples of meshes below.

* Generated mesh using builtin NGsolve mesh generator

* Several Hand made meshes including the Morgan Scott Mesh

* An imported mesh\*

\* User must select .vol file when prompted to upload a file. An example mesh can be downloaded from GitHub: https://github.com/armro/CheckStrangsConjecture$0














In [ ]:
#@title Uploading NGSolve
# Try direct pip install first.
!pip install -q ngsolve scipy numpy matplotlib

# Verify import. If this fails in your Colab runtime, use the fallback cell below.
import ngsolve
import netgen
from ngsolve.webgui import Draw
from IPython.display import display

print("NGSolve imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.9/30.9 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 14.5 MB/s eta 0:00:00


In [ ]:
#@title Main Code (make sure to run)
from ngsolve import *
from ngsolve import Draw, Redraw
from netgen.geom2d import unit_square
from netgen.geom2d import SplineGeometry
import numpy as np
import scipy.sparse as sp
import scipy.linalg as la
from IPython.display import display

from netgen.meshing import Mesh, MeshPoint, Element0D, Element1D, Element2D, FaceDescriptor
from netgen.csg import Pnt
from ngsolve import Mesh as NGS_Mesh
from math import tan
import math
from ngsolve.webgui import Draw

from google.colab import files

def singular_vertices(mesh):
  tol = 1e-10
  singular_verts = 0
  for v in mesh.vertices:
    meshv = mesh[v]

    if len(meshv.edges) == 4:

      #get other vertices for edges connected to v
      tan_vecs = []
      for ind, e in enumerate(meshv.edges):
        meshe = mesh[e]
        v1, v2 = meshe.vertices

        p1 = np.array(mesh[v1].point)
        p2 = np.array(mesh[v2].point)

        # compute unit tan vectors
        unit_tangent = p2-p1 /np.linalg.norm(p2-p1)
        tan_vecs.append(unit_tangent)

      # Make tan_vectors into 3-vectors to use numpy.cross()
      tan_3d = []
      for tv in tan_vecs:
        v_3d = np.pad(tv, (0,1))
        tan_3d.append(v_3d)

      # take cross product between all edges
      # cross product list for each veertex
      cross_prods = []
      colinear = []

      counter = 0
      while len(tan_3d)>1:
        for i in range(1, len(tan_3d)):
          cross = np.cross(tan_3d[0], tan_3d[i])
          if (np.absolute(cross) <= tol).all():
            #colinear.append(cross)
            counter += 1
        tan_3d.pop(0)

      if counter == 2:
        singular_verts +=1
  return singular_verts

def strang_holds(nullity):
  if nullity == 0:
    return True
  else:
    return False

def numerical_kernel(A, tol=1e-10):
    """
    Returns columns spanning the numerical right nullspace of A.
    That is, vectors c such that A c approx 0.
    """
    Ad = A.toarray()
    U, s, Vh = la.svd(Ad, full_matrices=True)

    rank = np.sum(s > tol)
    Z = Vh[rank:, :].T

    nullity = Z.shape[1]

    return nullity

# Assemble matrix that will determine if Strang's Conjecture holds
def assemble_strang_matrix(mesh, k, quad_order=None):
    """
    Assemble A_{ij} = int_Omega p_j div(v_i) dx

    V = VectorH1(mesh, order=k-1)  = [continuous P_{k-1}]^2
    Q = L2(mesh, order=k-2)        = discontinuous P_{k-2}

    Returns:
        A      scipy csr sparse matrix, shape (V.ndof, Q.ndof)
        mesh   NGSolve mesh
        V      vector H1 space
        Q      discontinuous L2 space
    """
    if k < 2:
        raise ValueError("Need k >= 2 because Q = L2(order=k-2).")

    Q = L2(mesh, order=k-2)
    V = VectorH1(mesh, order=k-1)

    # Product space ordered as Q first, V second.
    X = Q * V

    # Trial variables: (p, u)
    # Test variables:  (q, v)
    (p, u), (q, v) = X.TnT()

    bf = BilinearForm(X)
    #lf = LinearForm(X)

    if quad_order is None:
        bf += p * div(v) * dx
    else:
        bf += p * div(v) * dx(bonus_intorder=quad_order)

    bf.Assemble()

    rows, cols, vals = bf.mat.COO()
    B = sp.csr_matrix((vals, (rows, cols)), shape=(X.ndof, X.ndof))

    nq = Q.ndof
    nv = V.ndof

    # Because X = Q * V, the Q dofs are first and the V dofs are second.
    # The term p * div(v) has:
    #   trial column from Q,
    #   test row from V.
    A = B[nq:nq+nv, 0:nq].copy()

    sing_vertices = singular_vertices(mesh)

    #Nullity
    m = numerical_kernel(A, tol=1e-10)


    M = m - sing_vertices


    STR_H = strang_holds(M)

    return A, V, Q, sing_vertices, M, STR_H


In [ ]:
#@title Create Mesh: Generated Mesh (Using Built in Command in NGSOLVE)


genermesh = NGS_Mesh(unit_square.GenerateMesh(maxh=0.2))

Draw(genermesh)



In [ ]:
#@title Check Strang's Conjecture on  Generated Mesh

k=2 # polynomial size we are testing

mesh = genermesh  # providedmesh will be generated in an above cell


A, V, Q, sigma, M, strang = assemble_strang_matrix(mesh, k)


print("k =", k)
print(f"Number of singular vertices: {sigma}")
print(f"For this mesh Strang's Conjecture is: {strang}")

In [ ]:
#@title Create Mesh: Handmade Mesh #1 (Heart Mesh)
from netgen.meshing import Mesh as ntMesh
def heart_mesh():
  # 1. Initialize an empty 2D mesh
  netmesh = ntMesh(dim=2)

  # 2, Initialize coordinates for verticies
  points = [
      Pnt(-3.0, 4.0, 0.0),    #V0
      Pnt(-2.0, 4.0, 0.0),    #V1
      Pnt(-1.0, 4.0, 0.0),   #V2
      Pnt(1.0, 4.0, 0.0),   #V3
      Pnt(2.0, 4.0, 0.0),    #V4
      Pnt(3.0, 4.0, 0.0),    #V5
      Pnt(-3.0, 3.0, 0.0), #V6
      Pnt(0.0, 3.0, 0.0), #V7
      Pnt(3.0, 3.0, 0.0),  #V8
      Pnt(-3.0, 2.0, 0.0),  #V9
      Pnt(0.0, 2.0, 0.0),  #V10
      Pnt(3.0, 2.0, 0.0),  #V11
      Pnt(-1.0, 1.0, 0.0),  #V12
      Pnt(1.0, 1.0, 0.0),  #V13
      Pnt(0.0, -1.0, 0.0),  #V14
  ]
  # Add points/ vertices to mesh
  pids = []
  for p in points:
      pids.append(netmesh.Add(MeshPoint(p)))


  # 3. Define mesh regions (index 1 for geometry)
  netmesh.AddRegion("domain", dim=2)
  netmesh.Add(FaceDescriptor(surfnr=1, domin=1, bc=1))

  # 4. Define elements/triangles (e.g., creating a structured mesh of 4 triangles from the 6 points)
  # pids are indexed 0 through 14 based on the points list
  elements = [
      [pids[6], pids[1], pids[0]],              #T1
      [pids[2], pids[1], pids[6]],              #T2
      [pids[9], pids[2], pids[6]],              #T3
      [pids[7], pids[2], pids[9]],              #T4
      [pids[10], pids[7], pids[9]],             #T5
      [pids[12], pids[10], pids[9]],            #T6
      [pids[14], pids[12], pids[9]],            #T7
      [pids[14], pids[13], pids[12]],		#T8
      [pids[12], pids[13], pids[10]],		#T9
      [pids[11], pids[13], pids[14]],		#T10
      [pids[13], pids[11], pids[10]],		#T11
      [pids[11], pids[7], pids[10]],		#T12
      [pids[3], pids[7], pids[11]],		#T13
      [pids[8], pids[3], pids[11]],		#T14
      [pids[8], pids[4], pids[3]],		#T15
      [pids[5], pids[4], pids[8]],		#T16

  ]


  #Add triangles to mesh
  for el in elements:
      netmesh.Add(Element2D(1, el))

  # 5. Convert to NGSolve Mesh
  mesh = NGS_Mesh(netmesh)
  return mesh, points

heartmesh, p= heart_mesh()
Draw(heartmesh)


In [ ]:
#@title Check Strang's Conjecture on  Heart Mesh

k=2 # polynomial size we are testing

mesh = heartmesh  # providedmesh will be generated in an above cell


A, V, Q, sigma, M, strang = assemble_strang_matrix(mesh, k)


print("k =", k)
print(f"Number of singular vertices: {sigma}")
print(f"For this mesh Strang's Conjecture is: {strang}")

In [ ]:
#@title Create Mesh: Handmade Mesh #2 (A singular vertex present)
from netgen.meshing import Mesh as ntMesh
from google.colab import files
import ngsolve as ng
#Generate Morgan-scott mesh
#Generates equilateral triangle mesh with lengths = 2, centered at (0, tan(pi/3)/3)
def sing_vert_mesh():
  # 1. Initialize an empty 2D mesh
  netmesh = ntMesh(dim=2)
  #ngmesh.dim = 2

  # 2, Initialize coordinates for verticies
  points = [
      Pnt(1.0, 0.0, 0.0),    #V0
      Pnt(0.0, 1.0, 0.0),     #V1
      Pnt(-1.0, 0.0, 0.0),    #V2
      Pnt(0.0, -1.0, 0.0),     #V3
      Pnt(0.5, 0.0, 0.0),      #V4
      Pnt(0.0, 0.5, 0.0),      #V5
      Pnt(-0.5, 0.0, 0.0), #V6
      Pnt(0.0, -0.5, 0.0), #V7
      Pnt(0.0, 0.0, 0.0)  #V8

  ]
  # Add poiints/ vertices to mesh
  pids = []
  for p in points:
      pids.append(netmesh.Add(MeshPoint(p)))


  # 3. Define mesh regions (index 1 for geometry)
  netmesh.AddRegion("domain", dim=2)
  netmesh.Add(FaceDescriptor(surfnr=1, domin=1, bc=1))

  # 4. Define elements/triangles (e.g., creating a structured mesh of 4 triangles from the 6 points)
  # pids are indexed 0 through 5 based on the points list
  elements = [
      [pids[0], pids[1], pids[5]],              #T1
      [pids[1], pids[6], pids[5]],              #T2
      [pids[1], pids[2], pids[6]],              #T3
      [pids[2], pids[7], pids[6]],              #T4
      [pids[2], pids[3], pids[7]],              #T5
      [pids[3], pids[4], pids[7]],              #T6
      [pids[3], pids[0], pids[4]],              #T7
      [pids[4], pids[0], pids[5]],
      [pids[4], pids[5], pids[8]],
      [pids[5], pids[6], pids[8]],
      [pids[6], pids[7], pids[8]],
      [pids[7], pids[4], pids[8]]

  ]


  #Add triangles to mesh
  for el in elements:
      netmesh.Add(Element2D(1, el))

  # Save mesh
 # netmesh.Save("sing_vert_mesh.vol")
 # files.download("sing_vert_mesh.vol")
  # 5. Convert to NGSolve Mesh
  mesh = NGS_Mesh(netmesh)

  return mesh

handmade2= sing_vert_mesh()
Draw(handmade2)



In [ ]:
#@title Check Strang's Conjecture on  Hand Made mesh # 2

k=2 # polynomial size we are testing

mesh = handmade2  # providedmesh will be generated in an above cell

A, V, Q, sigma, M, strang = assemble_strang_matrix(mesh, k)

print("k =", k)
print(f"Number of singular vertices: {sigma}")
print(f"For this mesh Strang's Conjecture is: {strang}")

In [ ]:
#@title Create Mesh: Hand Made mesh # 3 (Morgan-Scott)
def morgan_scott_mesh():
  # Initialize an empty 2D mesh
  ngmesh = Mesh()
  ngmesh.dim = 2

  #Initialize vertices of mesh
  points = [
      Pnt(-1.0, 0.0, 0.0),                                            #V0
      Pnt(1.0, 0.0, 0.0),                                             #V1
      Pnt(0.0, tan((np.pi)/3), 0.0),                                  #V2
      Pnt(-1/3, math.sqrt(2)/6 +4*math.sqrt(6)/18, 0.0),    #V3
      Pnt(0, math.sqrt(6)/18, 0.0),          #V4
      Pnt(1/3, math.sqrt(2)/6 + 4*math.sqrt(6)/18, 0.0)       #V5

  ]


  # Add modified vertices to mesh
  pids = []
  for p in points:
    pids.append(ngmesh.Add(MeshPoint(p)))


  # Define mesh regions (index 1 for geometry)
  ngmesh.AddRegion("domain", dim=2)
  ngmesh.Add(FaceDescriptor(surfnr=1, domin=1, bc=1))

  # Define elements (triangles)
  elements = [
      [pids[0], pids[1], pids[4]],                #T1
      [pids[1], pids[5], pids[4]],                #T2
      [pids[1], pids[2], pids[5]],                #T3
      [pids[5], pids[2], pids[3]],                #T4
      [pids[2], pids[0], pids[3]],                #T5
      [pids[4], pids[3], pids[0]],                #T6
      [pids[5], pids[3], pids[4]]                 #T7
  ]

  # Add triangles to mesh
  for el in elements:
      ngmesh.Add(Element2D(1, el))

  # Convert to NGSolve Mesh
  mesh = NGS_Mesh(ngmesh)

  return mesh, points

msmesh,p= morgan_scott_mesh()
Draw(msmesh)

In [ ]:
#@title Check Strang's Conjecture on  Hand Made mesh # 3 (Morgan-Scott)

k=2 # polynomial size we are testing

mesh = msmesh  # providedmesh will be generated in an above cell


A, V, Q, sigma, M, strang = assemble_strang_matrix(mesh, k)


print("k =", k)
print(f"Number of singular vertices: {sigma}")
print(f"For this mesh Strang's Conjecture is: {strang}")

In [ ]:
#@title Create Mesh: Imported Mesh
from google.colab import files
from netgen.meshing import Mesh as ntMesh

# "sing_vert_mesh.vol" is a mesh that needs to be in your directory
# it can be downloaded from GitHub
# https://github.com/armro/CheckStrangsConjecture/upload/main$0
# Select "sing_vert_mesh.vol" from your directory when prompted

fileupload = files.upload()
mesh_name=list(fileupload.keys())[0]

netmesh = ntMesh()
netmesh.Load(mesh_name)
importedmesh = ngsolve.Mesh(netmesh)

Draw(importedmesh)

In [ ]:
#@title Check Strang's Conjecture on  Imported Mesh

k=2 # polynomial size we are testing

mesh = importedmesh  # providedmesh will be generated in an above cell


A, V, Q, sigma, M, strang = assemble_strang_matrix(mesh, k)


print("k =", k)
print(f"Number of singular vertices: {sigma}")
print(f"For this mesh Strang's Conjecture is: {strang}")